[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · **[03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb)** · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 2 课：State 与 Reducer](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) ｜ [➡️ 下一课：第 4 课：手写 ReAct Agent](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb)

---

# 第 3 课：控制流 —— 条件边、循环、并行、Send(map-reduce)、Command

LangGraph 的执行模型来自 Google Pregel / BSP：
- 按 "super-step（超步）" 推进。每个超步里，所有被触发的节点"并行"执行；
- 超步结束时，所有更新通过 reducer 合并进 state，然后决定下一批节点；
- 没有节点被触发时，图结束。

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
import operator

from typing_extensions import Annotated, Literal, TypedDict

from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send

In [3]:
# A. 条件边 + 循环：猜数字

In [4]:
class GuessState(TypedDict):
    target: int
    low: int
    high: int
    guess: int
    tries: Annotated[list[int], operator.add]


def make_guess(state: GuessState):
    g = (state["low"] + state["high"]) // 2
    return {"guess": g, "tries": [g]}


def adjust(state: GuessState):
    if state["guess"] < state["target"]:
        return {"low": state["guess"] + 1}
    return {"high": state["guess"] - 1}


def route(state: GuessState) -> Literal["adjust", "__end__"]:
    # 路由函数返回 "下一个节点名"（或节点名列表 / Send 列表）
    return END if state["guess"] == state["target"] else "adjust"


guess_graph = (
    StateGraph(GuessState)
    .add_node(make_guess).add_node(adjust)
    .add_edge(START, "make_guess")
    .add_conditional_edges("make_guess", route)
    .add_edge("adjust", "make_guess")  # 形成循环
    .compile()
)

In [5]:
# B. 静态并行（fan-out / fan-in）

In [6]:
class ParState(TypedDict):
    results: Annotated[list[str], operator.add]  # 并行写同一键 -> 必须有 reducer


def mk(name):
    def node(state):
        return {"results": [name]}
    return node


par_graph = (
    StateGraph(ParState)
    .add_node("a", mk("a")).add_node("b", mk("b")).add_node("c", mk("c")).add_node("join", mk("join"))
    .add_edge(START, "a")
    .add_edge("a", "b").add_edge("a", "c")       # b 和 c 在同一个超步并行
    .add_edge(["b", "c"], "join")                 # 等 b、c 都完成才执行 join
    .compile()
)


class BadState(TypedDict):
    results: list[str]  # 没有 reducer


bad_graph = (
    StateGraph(BadState)
    .add_node("b", lambda s: {"results": ["b"]}).add_node("c", lambda s: {"results": ["c"]})
    .add_edge(START, "b").add_edge(START, "c")
    .compile()
)

In [7]:
# C. Send：动态 map-reduce（运行时才知道要分几份）

In [8]:
class MapState(TypedDict):
    topics: list[str]
    jokes: Annotated[list[str], operator.add]


class JokeInput(TypedDict):
    topic: str


def fan_out(state: MapState):
    # 返回 Send 列表：每个 Send 用"自定义的输入"调用一次目标节点
    return [Send("write_joke", {"topic": t}) for t in state["topics"]]


def write_joke(state: JokeInput):
    return {"jokes": [f"关于{state['topic']}的笑话 🤡"]}


map_graph = (
    StateGraph(MapState)
    .add_node(write_joke)
    .add_conditional_edges(START, fan_out, ["write_joke"])
    .compile()
)

In [9]:
# D. Command：在节点里"同时"更新状态 + 决定跳转（无需写条件边）

In [10]:
class TicketState(TypedDict):
    text: str
    handled_by: str


def triage(state: TicketState) -> Command[Literal["billing", "tech"]]:
    dest = "billing" if "退款" in state["text"] else "tech"
    return Command(update={"handled_by": f"triage->{dest}"}, goto=dest)


def billing(state: TicketState):
    return {"handled_by": state["handled_by"] + " ✅billing"}


def tech(state: TicketState):
    return {"handled_by": state["handled_by"] + " ✅tech"}


cmd_graph = (
    StateGraph(TicketState)
    .add_node(triage).add_node(billing).add_node(tech)
    .add_edge(START, "triage")
    .compile()
)

## ▶️ 运行实验

In [11]:
out = guess_graph.invoke({"target": 73, "low": 0, "high": 100, "tries": []})
print("A. 猜数字过程:", out["tries"])

try:
    guess_graph.invoke({"target": 73, "low": 0, "high": 100, "tries": []}, {"recursion_limit": 4})
except GraphRecursionError as e:
    print("A'. recursion_limit=4 触发:", type(e).__name__)

print("B. 并行:", par_graph.invoke({"results": []})["results"])
for step in par_graph.stream({"results": []}, stream_mode="updates"):
    print("   超步更新:", step)
try:
    bad_graph.invoke({"results": []})
except InvalidUpdateError as e:
    print("B'. 无 reducer 并行写入报错:", type(e).__name__)

print("C. Send:", map_graph.invoke({"topics": ["猫", "程序员", "咖啡"], "jokes": []})["jokes"])
print("D. Command:", cmd_graph.invoke({"text": "我要退款", "handled_by": ""}))
print(cmd_graph.get_graph().draw_mermaid())

A. 猜数字过程: [50, 75, 62, 68, 71, 73]
A'. recursion_limit=4 触发: GraphRecursionError
B. 并行: ['a', 'b', 'c', 'join']
   超步更新: {'a': {'results': ['a']}}
   超步更新: {'b': {'results': ['b']}}
   超步更新: {'c': {'results': ['c']}}
   超步更新: {'join': {'results': ['join']}}
B'. 无 reducer 并行写入报错: InvalidUpdateError
C. Send: ['关于猫的笑话 🤡', '关于程序员的笑话 🤡', '关于咖啡的笑话 🤡']
D. Command: {'text': '我要退款', 'handled_by': 'triage->billing ✅billing'}
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	billing(billing)
	tech(tech)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	triage -.-> billing;
	triage -.-> tech;
	billing --> __end__;
	tech --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## ✏️ 练习：
 1. 给 B 加一个 "d" 节点，路径 a->b->d->join 与 a->c->join，join 会执行几次？
    （试试 add_node("join", ..., defer=True) 的效果）
 2. 用 Send 实现：把一段长文本切块 -> 并行统计字数 -> 汇总。

In [12]:
# 在这里写你的练习代码 👇


In [13]:
#@title 💡 展开查看参考答案与解析（可点击 ▶ 直接运行验证） { display-mode: "form" }
# 练习 1：不等长分支 a->b->d->join 与 a->c->join，对比普通节点 vs defer=True
def build_asym(defer_join: bool):
    return (
        StateGraph(ParState)
        .add_node("a", mk("a")).add_node("b", mk("b")).add_node("c", mk("c")).add_node("d", mk("d"))
        .add_node("join", mk("join"), defer=defer_join)
        .add_edge(START, "a")
        .add_edge("a", "b").add_edge("a", "c")
        .add_edge("b", "d")
        .add_edge("d", "join").add_edge("c", "join")
        .compile()
    )

print("练习 1 (defer=False，join 被触发 2 次):", build_asym(False).invoke({"results": []})["results"])
print("练习 1 (defer=True ，join 等所有分支结束只跑 1 次):", build_asym(True).invoke({"results": []})["results"])

# 练习 2：用 Send 实现长文本切块 -> 并行统计字数 -> 汇总
class WordCountState(TypedDict):
    text: str
    counts: Annotated[list[int], operator.add]
    total: int

def split_to_sends(state: WordCountState):
    chunks = [line.strip() for line in state["text"].split("。") if line.strip()]
    return [Send("count_chunk", {"chunk": c}) for c in chunks]

def count_chunk(state: dict):
    return {"counts": [len(state["chunk"])]}

wc_graph = (
    StateGraph(WordCountState)
    .add_node("count_chunk", count_chunk)
    .add_node("sum_up", lambda s: {"total": sum(s["counts"])})
    .add_conditional_edges(START, split_to_sends, ["count_chunk"])
    .add_edge("count_chunk", "sum_up")
    .compile()
)
print("练习 2 Map-Reduce 字数统计:", wc_graph.invoke({"text": "LangGraph很强大。它支持并行。还能持久化。", "counts": []}))

练习 1 (defer=False，join 被触发 2 次): ['a', 'b', 'c', 'd', 'join', 'join']
练习 1 (defer=True ，join 等所有分支结束只跑 1 次): ['a', 'b', 'c', 'd', 'join']
练习 2 Map-Reduce 字数统计: {'text': 'LangGraph很强大。它支持并行。还能持久化。', 'counts': [12, 5, 5], 'total': 22}


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · **[03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb)** · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 2 课：State 与 Reducer](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) ｜ [➡️ 下一课：第 4 课：手写 ReAct Agent](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb)

---